# MedArk3D v5

Runtime → Change runtime type → **T4 GPU**, then Run All. ~1 h at 32³.

### What changed after the 60-cycle v4 run

The v4 run worked — `zstd` climbed 0.0006 → 0.0028 and `cons` 0.0004 → 0.03, so the
projector is alive and the consistency loss is doing work for the first time. But cell 9
only saved `last.pt`, and the final cycle is not the best cycle:

| | peak | @cycle | final | lost |
|---|---|---|---|---|
| organ | 0.989 | 44 | 0.989 | — |
| nodule | 0.886 | 29 | 0.855 | −0.031 |
| **fracture** | **0.662** | **22** | **0.587** | **−0.075** |
| adrenal | 0.783 | 60 | 0.783 | — |
| vessel | 0.877 | 53 | 0.861 | −0.016 |
| synapse | 0.735 | 58 | 0.735 | — |

Mean of per-dataset peaks **0.8220** vs final-cycle **0.8017** — 0.020 thrown away, and
fracture alone lost 0.075. The peaks span cycle 22 to 60, so no single global checkpoint
can serve all six. v5 keeps a **best-per-dataset teacher snapshot**, selected on
validation and reported on test (standard per-task early stopping), and probe and
fine-tune each load their own dataset's best.

Also: `CYCLES = 40` so the cosine schedule actually anneals to zero instead of being
truncated; **class-weighted** multi-class loss computed from the label counts (a no-op
when balanced); and the collapse threshold corrected — healthy `zstd` for an
L2-normalised 512-d embedding is ~1/√512 ≈ 0.044, so 1e-3 was mis-calibrated alarmism.

## 1. Install

In [ ]:
!pip -q install medmnist monai einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.9 MB/s eta 0:00:00


## 2. Config

`RES = 64` is the one principled upgrade left. `window_size=7` is fixed by the SSL
checkpoint, so at 32³ the feature maps shrink below the window and MONAI pads them into
a single window — the attention goes global and only 2 of 5 stages keep locality. At 64³
it is 3 of 5, matching the regime the SSL weights were pretrained in. Cell 5 prints the
table. Cost is roughly 4-6× the time per cycle; run 32³ first, then decide.

In [ ]:
import os, copy, json, math, time, hashlib, subprocess
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

NATIVE, RES = 28, 32      # native 28^3 volumes, model input 32^3   (~35 s/cycle on a T4)
# NATIVE, RES = 64, 64    # the locality upgrade - see cell 5       (~3 min/cycle)

BATCH       = 32
CYCLES      = 40          # cosine anneals to zero exactly here; v4's 60 was truncated
SOLO_EPOCHS = CYCLES      # matched budget - do not decouple
MU          = 0.9         # per-DATASET-EPOCH cadence: 6*CYCLES updates, 0.9^240 ~ 0
LAM         = 1.0
SEED        = 42
ZSTD_FLOOR  = 5e-3        # healthy is ~1/sqrt(512) = 0.044; below this = real collapse

CFG = dict(arch="ssl", pool="maxavg34", head_on="proj")

OUT = "/content/medark3d_v5"
try:
    from google.colab import drive; drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/medark3d_v5"
except Exception:
    pass
os.makedirs(OUT, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if dev.type == "cuda": print(torch.cuda.get_device_properties(0).name)
print("out:", OUT)

Mounted at /content/drive
Tesla T4
out: /content/drive/MyDrive/medark3d_v5


## 3. Tasks

In [ ]:
import medmnist
from medmnist import INFO
from torch.utils.data import Dataset, DataLoader

FLAGS = ["organmnist3d", "nodulemnist3d", "fracturemnist3d",
         "adrenalmnist3d", "vesselmnist3d", "synapsemnist3d"]
# Official MedMNIST ResNet-18+3D test AUC. These are FULLY TRAINED SUPERVISED models,
# so the fair comparison is cell 12's fine-tune, not cell 11's frozen probe.
BASELINE = dict(organmnist3d=.996, nodulemnist3d=.863, fracturemnist3d=.712,
                adrenalmnist3d=.827, vesselmnist3d=.874, synapsemnist3d=.820)

SPECS = []
for i, f in enumerate(FLAGS):
    n = len(INFO[f]["label"])          # INFO is authoritative; the methodology table is wrong
    SPECS.append(dict(id=i, flag=f, cls=INFO[f]["python_class"], n=n,
                      binary=(n == 2), out=(1 if n == 2 else n)))
for s in SPECS:
    print(f"{s['id']}  {s['flag']:<18} {s['n']}cls  "
          f"{'binary' if s['binary'] else 'multiclass':<11} baseline {BASELINE[s['flag']]:.3f}")

0  organmnist3d       11cls  multiclass  baseline 0.996
1  nodulemnist3d      2cls  binary      baseline 0.863
2  fracturemnist3d    3cls  multiclass  baseline 0.712
3  adrenalmnist3d     2cls  binary      baseline 0.827
4  vesselmnist3d      2cls  binary      baseline 0.874
5  synapsemnist3d     2cls  binary      baseline 0.820


### 3a. Prefetch — Zenodo 504s under load and medmnist quits on the first one

In [ ]:
ROOT = os.path.expanduser("~/.medmnist"); os.makedirs(ROOT, exist_ok=True)
SFX = "" if NATIVE == 28 else f"_{NATIVE}"

def md5_ok(p, want):
    if not os.path.exists(p): return False
    h = hashlib.md5()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""): h.update(c)
    return h.hexdigest() == want

for f in FLAGS:
    p, want, url = f"{ROOT}/{f}{SFX}.npz", INFO[f][f"MD5{SFX}"], INFO[f][f"url{SFX}"]
    if md5_ok(p, want): print("cached ", f); continue
    for a in range(1, 6):
        subprocess.run(["wget","-q","--tries=5","--waitretry=15","--timeout=180","-O",p,url])
        if md5_ok(p, want): print("ok     ", f); break
        if os.path.exists(p): os.remove(p)
        print(f"retry   {f} ({a}/5)", flush=True)
    else:
        raise RuntimeError(f"{f} failed. Get {url} manually into {ROOT}/")
print("archives verified")

ok      organmnist3d
ok      nodulemnist3d
ok      fracturemnist3d
ok      adrenalmnist3d
ok      vesselmnist3d
ok      synapsemnist3d
archives verified


### 3b. Data

Each (dataset, split) is built **once**. `num_workers=0`: augmentation is on the GPU, so
`__getitem__` only slices a uint8 array — worker processes would add IPC and a RAM copy
for zero parallel work, which is what OOM-killed an earlier version.

Loss weights are computed from the actual label counts, so they are a no-op on a balanced
dataset and only bite where the prior is skewed.

In [ ]:
class Vol3D(Dataset):
    def __init__(self, spec, split):
        ds = getattr(medmnist, spec["cls"])(split=split, download=True, size=NATIVE)
        self.x = np.ascontiguousarray(np.asarray(ds.imgs))
        self.y = np.asarray(ds.labels).ravel().astype(np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return torch.from_numpy(self.x[i])[None], self.y[i]

DS, STATS, POSW, CLSW = {}, {}, {}, {}
for s in SPECS:
    for sp in ("train", "val", "test"): DS[(s["id"], sp)] = Vol3D(s, sp)
    tr = DS[(s["id"], "train")]
    STATS[s["id"]] = (float(tr.x.mean())/255., float(tr.x.std())/255. + 1e-8)
    cnt = np.bincount(tr.y, minlength=s["n"]).astype(np.float32)
    if s["binary"]:
        POSW[s["id"]] = torch.tensor(cnt[0]/max(cnt[1], 1), device=dev)
        note = f"pos_weight {POSW[s['id']].item():.2f}"
    else:
        w = len(tr.y) / (s["n"] * np.maximum(cnt, 1))
        CLSW[s["id"]] = torch.tensor(w, device=dev)
        note = f"class_w {np.round(w, 2)}"
    print(f"{s['flag']:<18} train {len(tr):5d}  counts {cnt.astype(int)}  {note}")
print(f"resident {sum(d.x.nbytes for d in DS.values())/2**30:.2f} GB")

def loaders(spec, batch=BATCH):
    mk = lambda sp, sh: DataLoader(DS[(spec["id"], sp)], batch_size=batch, shuffle=sh,
                                   num_workers=0, pin_memory=(dev.type == "cuda"))
    return mk("train", True), mk("val", False), mk("test", False)

TR, VA, TE = {}, {}, {}
for s in SPECS: TR[s["id"]], VA[s["id"]], TE[s["id"]] = loaders(s)

organmnist3d       train   971  counts [115 115 115  91  93  94  40  39  39 115 115]  class_w [0.77 0.77 0.77 0.97 0.95 0.94 2.21 2.26 2.26 0.77 0.77]
nodulemnist3d      train  1158  counts [863 295]  pos_weight 2.93
fracturemnist3d    train  1027  counts [473 383 171]  class_w [0.72 0.89 2.  ]
adrenalmnist3d     train  1188  counts [929 259]  pos_weight 3.59
vesselmnist3d      train  1335  counts [1185  150]  pos_weight 7.90
synapsemnist3d     train  1230  counts [331 899]  pos_weight 0.37
resident 0.20 GB


## 4. GPU preprocessing and augmentation

Rotation, spatial scaling, axis flips, intensity scale/shift — batched on the GPU.
Spatial scaling matters for vessel and adrenal: those are voxelised meshes, essentially
binary masks, so intensity augmentation on them is a no-op.

In [ ]:
def to_gpu(x, sid):
    x = x.to(dev, non_blocking=True).float().div_(255.)
    if x.shape[-1] != RES:
        x = F.interpolate(x, size=(RES,)*3, mode="trilinear", align_corners=False)
    m, s = STATS[sid]; return (x - m) / s

def _rot(B, rad, device):
    a = (torch.rand(B,3,device=device)*2-1)*rad
    ca, sa = torch.cos(a), torch.sin(a)
    z, o = torch.zeros(B,device=device), torch.ones(B,device=device)
    Rx = torch.stack([o,z,z, z,ca[:,0],-sa[:,0], z,sa[:,0],ca[:,0]],1).view(B,3,3)
    Ry = torch.stack([ca[:,1],z,sa[:,1], z,o,z, -sa[:,1],z,ca[:,1]],1).view(B,3,3)
    Rz = torch.stack([ca[:,2],-sa[:,2],z, sa[:,2],ca[:,2],z, z,z,o],1).view(B,3,3)
    return Rz @ Ry @ Rx

def augment(x, rad=.26, zoom=.15, flip_p=.5, scale=.15, shift=.1):
    B, d = x.size(0), x.device
    for ax in (2,3,4):
        m = torch.rand(B, device=d) < flip_p
        x = torch.where(m.view(-1,1,1,1,1), x.flip(ax), x)
    s = 1 + (torch.rand(B,1,1,device=d)*2-1)*zoom
    th = torch.cat([_rot(B, rad, d)*s, torch.zeros(B,3,1,device=d)], 2)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False),
                      align_corners=False, padding_mode="zeros")
    r = lambda: (torch.rand(B,1,1,1,1,device=d)*2-1)
    return x*(1 + r()*scale) + r()*shift

_t = torch.randn(4,1,RES,RES,RES)
# atol 1e-4, not exact: an identity grid_sample carries ~4e-6 float resampling error
assert augment(_t, 0,0,0,0,0).allclose(_t, atol=1e-4), "zero-augment must be identity"
assert not augment(_t).allclose(_t) and torch.isfinite(augment(_t)).all()
print("augmentation OK")

augmentation OK


## 5. Model

In [ ]:
from monai.networks.nets.swin_unetr import SwinTransformer

# The "ssl" config is dictated by the checkpoint, not chosen. Any deviation drops
# tensors silently, so the load count is asserted.
ARCH = {
  "ssl": dict(embed=48, win=7, depths=(2,2,2,2), nh=(3,6,12,24), pre=True,
              opt="adamw", lr=3e-4, enc_mult=0.1, wd=1e-2),
  "v1":  dict(embed=24, win=2, depths=(2,2,6,2), nh=(3,6,12,24), pre=False,
              opt="sgd",   lr=1e-2, enc_mult=1.0, wd=1e-4),
}
SSL_URL = ("https://github.com/Project-MONAI/MONAI-extra-test-data/"
           "releases/download/0.8.1/model_swinvit.pt")

def build_enc(a):
    enc = SwinTransformer(in_chans=1, embed_dim=a["embed"], window_size=(a["win"],)*3,
        patch_size=(2,2,2), depths=a["depths"], num_heads=a["nh"], mlp_ratio=4.,
        qkv_bias=True, drop_path_rate=0.0, norm_layer=nn.LayerNorm,
        patch_norm=False, use_checkpoint=False, spatial_dims=3)
    if not a["pre"]: return enc
    if not os.path.exists("swinvit.pt"):
        subprocess.run(["wget","-q","--tries=5","-O","swinvit.pt",SSL_URL], check=True)
    sd = torch.load("swinvit.pt", map_location="cpu", weights_only=False)["state_dict"]
    sd = {k.replace("module.","",1).replace("mlp.fc1","mlp.linear1")
           .replace("mlp.fc2","mlp.linear2"): v for k,v in sd.items()}
    own = enc.state_dict()
    keep = {k:v for k,v in sd.items() if k in own and own[k].shape == v.shape}
    assert len(keep) == len(own), f"loaded {len(keep)}/{len(own)} - config mismatch"
    enc.load_state_dict(keep, strict=True); return enc


class MedArk3D(nn.Module):
    def __init__(self, outs, arch="ssl", pool="maxavg34", head_on="proj", pdim=512):
        super().__init__()
        self.a = ARCH[arch]; self.pool, self.head_on = pool, head_on
        self.enc = build_enc(self.a)
        with torch.no_grad():
            st = self.enc(torch.zeros(1,1,RES,RES,RES), normalize=True)
        self.idx = (4,) if pool == "avg4" else (3,4)
        mult = 1 if pool == "avg4" else 2
        self.d = sum(mult*st[i].shape[1] for i in self.idx)
        self.norm = nn.LayerNorm(self.d)
        self.proj = nn.Sequential(nn.Linear(self.d,self.d), nn.LayerNorm(self.d),
                                  nn.GELU(), nn.Linear(self.d, pdim))
        self.heads = nn.ModuleList([nn.Linear(self.d if head_on=="feat" else pdim, o)
                                    for o in outs])

    def feat(self, x):
        h = self.enc(x, normalize=True); v = []
        for i in self.idx:
            v.append(F.adaptive_avg_pool3d(h[i],1).flatten(1))
            if self.pool != "avg4":         # max keeps a sparse local activation alive
                v.append(F.adaptive_max_pool3d(h[i],1).flatten(1))
        return self.norm(torch.cat(v,1))

    def forward(self, x, tid=None):
        f = self.feat(x)
        p = self.proj(f)                    # UNNORMALISED -> heads; logits must be free
        z = F.normalize(p, dim=-1)          # normalised   -> consistency loss only
        if tid is None: return f, p, z
        return f, p, z, self.heads[tid](f if self.head_on == "feat" else p)

    def make_opt(self, lr=None, enc_mult=None):
        a = self.a; lr = lr or a["lr"]; em = a["enc_mult"] if enc_mult is None else enc_mult
        new = (list(self.norm.parameters()) + list(self.proj.parameters())
               + list(self.heads.parameters()))
        g = [{"params": list(self.enc.parameters()), "lr": lr*em},
             {"params": new, "lr": lr}]
        return (torch.optim.SGD(g, momentum=0.9, weight_decay=a["wd"], nesterov=True)
                if a["opt"] == "sgd" else torch.optim.AdamW(g, weight_decay=a["wd"]))


_m = MedArk3D([s["out"] for s in SPECS], **CFG)
_st = _m.enc(torch.zeros(1,1,RES,RES,RES), normalize=True)
_f = [t.shape[2] for t in _st]; _w = [max(1, -(-s // ARCH[CFG["arch"]]["win"])) for s in _f]
print(f"feature maps  {_f}")
print(f"windows/axis  {_w}   -> stages with LOCAL attention: {sum(1 for w in _w if w>1)}/5"
      f"   (SSL pretraining regime = 3/5)")
print(f"feat_dim {_m.d} | params {sum(p.numel() for p in _m.parameters()):,}")
_m.zero_grad(); _m(torch.randn(2,1,RES,RES,RES), 0)[3].sum().backward()
assert _m.proj[0].weight.grad.abs().sum() > 0, "projector has no classification gradient"
print("projector receives classification gradient: OK")
del _m, _st

feature maps  [16, 8, 4, 2, 1]
windows/axis  [3, 2, 1, 1, 1]   -> stages with LOCAL attention: 2/5   (SSL pretraining regime = 3/5)
feat_dim 2304 | params 14,571,332
projector receives classification gradient: OK


## 6. Losses and metrics

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, matthews_corrcoef

class EMA:
    def __init__(self, model, mu=MU):
        self.mu = mu; self.t = copy.deepcopy(model).eval()
        for p in self.t.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for tp, sp in zip(self.t.parameters(), model.parameters()):
            tp.mul_(self.mu).add_(sp.detach(), alpha=1-self.mu)

def cls_loss(lg, y, spec):
    # weights come from the label counts, so this is a no-op on a balanced dataset
    if spec["binary"]:
        return F.binary_cross_entropy_with_logits(
            lg.squeeze(-1).float(), y.float(), pos_weight=POSW[spec["id"]])
    return F.cross_entropy(lg.float(), y.long(), weight=CLSW[spec["id"]])

def cons_loss(zs, zt):
    return (zs.float()-zt.float()).pow(2).sum(-1).mean()   # squared L2, not mse_loss/512

def ovr_auc(y, p, n):
    a = [roc_auc_score((y==c).astype(int), p[:,c])
         for c in range(n) if 0 < (y==c).sum() < len(y)]
    return float(np.mean(a)) if a else float("nan")

@torch.no_grad()
def score(model, loader, spec, head=None, tta=False, full=False):
    model.eval(); head = spec["id"] if head is None else head
    P, Y, Z = [], [], []
    for xb, y in loader:
        x = to_gpu(xb, spec["id"])
        views = [x, x.flip(2), x.flip(3), x.flip(4)] if tta else [x]
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type=="cuda"):
            outs = [model(v, head) for v in views]
        lg = torch.stack([o[3].float() for o in outs]).mean(0)
        P.append((torch.sigmoid(lg.squeeze(-1)) if spec["binary"]
                  else torch.softmax(lg,-1)).cpu().numpy())
        Y.append(y.numpy()); Z.append(outs[0][2].float().cpu().numpy())
    p, y, z = np.concatenate(P), np.concatenate(Y), np.concatenate(Z)
    o = dict(auc=float("nan"), zstd=float(z.std(0).mean()))
    if len(np.unique(y)) < 2: return o
    if spec["binary"]:
        o["auc"] = float(roc_auc_score(y,p))
        fpr,tpr,thr = roc_curve(y,p); pred = (p >= thr[int(np.argmax(tpr-fpr))]).astype(int)
    else:
        o["auc"] = ovr_auc(y,p,spec["n"]); pred = p.argmax(1)
    if full:                                 # only at final eval; the loops read auc
        o.update(acc=float((pred==y).mean()),
                 f1=float(f1_score(y,pred,average="macro",zero_division=0)),
                 mcc=float(matthews_corrcoef(y,pred)))
    return o

## 7. Single-dataset trainer

Used by cell 8 (solo baselines, SSL init) and cell 12 (fine-tuning, Ark-teacher init).
`init` is the only difference between the two protocols.

In [ ]:
def train_one(spec, epochs, init=None, lr=None, enc_mult=None, log=False):
    torch.manual_seed(SEED)
    tr, va, te = TR[spec["id"]], VA[spec["id"]], TE[spec["id"]]
    m = MedArk3D([spec["out"]], **CFG).to(dev)
    if init is not None:
        sd = {k: v for k, v in init.items() if not k.startswith("heads.")}
        got = m.load_state_dict(sd, strict=False)
        assert not [k for k in got.missing_keys if not k.startswith("heads.")]
    opt = m.make_opt(lr=lr, enc_mult=enc_mult)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in opt.param_groups],
        total_steps=epochs*len(tr), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type=="cuda")
    best, best_sd = -1, None
    for ep in range(epochs):
        m.train()
        for xb, y in tr:
            x = augment(to_gpu(xb, spec["id"])); y = y.to(dev, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type=="cuda"):
                loss = cls_loss(m(x,0)[3], y, spec)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 5.)
            scaler.step(opt); scaler.update(); sch.step()
        v = score(m, va, spec, head=0)["auc"]
        if v > best: best, best_sd = v, copy.deepcopy(m.state_dict())
        if log: print(f"    ep{ep+1} val {v:.4f}", flush=True)
    m.load_state_dict(best_sd)
    r = score(m, te, spec, head=0, tta=True, full=True); r["val"] = best
    del m, opt; torch.cuda.empty_cache()
    return r

## 8. Solo baselines

Same architecture, same budget as the cyclic run, one dataset each, no teacher and no
consistency loss. This is the control that `probe − solo` and `ft − solo` measure against.

In [ ]:
SOLO = {}
for s in SPECS:
    t0 = time.time(); SOLO[s["flag"]] = train_one(s, SOLO_EPOCHS)
    r = SOLO[s["flag"]]
    print(f"{s['flag']:<18} test AUC {r['auc']:.4f}  (base {BASELINE[s['flag']]:.3f})  "
          f"ACC {r['acc']:.4f}  [{time.time()-t0:.0f}s]", flush=True)
json.dump(SOLO, open(f"{OUT}/solo.json","w"), indent=2)
print(f"\nsolo mean {np.nanmean([v['auc'] for v in SOLO.values()]):.4f}  "
      f"(baseline mean {np.mean(list(BASELINE.values())):.4f})")

organmnist3d       test AUC 0.9713  (base 0.996)  ACC 0.7246  [705s]
nodulemnist3d      test AUC 0.8662  (base 0.863)  ACC 0.7677  [828s]
fracturemnist3d    test AUC 0.6149  (base 0.712)  ACC 0.4292  [721s]
adrenalmnist3d     test AUC 0.8008  (base 0.827)  ACC 0.7550  [829s]
vesselmnist3d      test AUC 0.7826  (base 0.874)  ACC 0.5183  [953s]
synapsemnist3d     test AUC 0.7066  (base 0.820)  ACC 0.6903  [881s]

solo mean 0.7904  (baseline mean 0.8487)


## 9. Cyclic pretraining

One epoch per dataset, six per cycle, EMA teacher updated after each dataset epoch.

**Per-dataset best snapshot.** v4's peaks landed on cycles 22 through 60, so one global
checkpoint cannot serve all six datasets. Each dataset's best teacher (selected on
validation) is kept in RAM and written to `best_<flag>.pt` only when it improves, so the
Drive write cost stays low. Cells 11 and 12 load each dataset's own best.

In [ ]:
student = MedArk3D([s["out"] for s in SPECS], **CFG).to(dev)
teacher = EMA(student); teacher.t.to(dev)
opt = student.make_opt()
scaler = torch.amp.GradScaler("cuda", enabled=dev.type=="cuda")
spc = sum(len(TR[s["id"]]) for s in SPECS)
warm, tot = spc, CYCLES*spc
# clamped: a resume with a changed CYCLES degrades to a flat LR instead of crashing
sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda i: (
    (i+1)/warm if i < warm else
    .5*(1+math.cos(math.pi*min(1., (i-warm)/max(1, tot-warm))))))

def snap(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

start, hist, ck = 0, [], f"{OUT}/last.pt"
BEST_AUC = {s["flag"]: -1.0 for s in SPECS}
if os.path.exists(ck):
    st = torch.load(ck, map_location=dev, weights_only=False)
    student.load_state_dict(st["s"]); teacher.t.load_state_dict(st["t"])
    opt.load_state_dict(st["o"]); sch.load_state_dict(st["sch"]); scaler.load_state_dict(st["sc"])
    start, hist = st["cycle"], st["hist"]; BEST_AUC.update(st.get("best_auc", {}))
    print("resumed at cycle", start)

for cyc in range(start, CYCLES):
    t0 = time.time()
    for s in SPECS:
        student.train(); teacher.t.eval(); agg, n = np.zeros(3), 0
        for xb, y in TR[s["id"]]:
            xt = to_gpu(xb, s["id"]); xs = augment(xt)
            y = y.to(dev, non_blocking=True)
            with torch.no_grad():                 # teacher fp32: fp16 here gave NaN
                zt = teacher.t(xt)[2]
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type=="cuda"):
                _, _, zs, lg = student(xs, s["id"])
                lc = cls_loss(lg, y, s); lm = cons_loss(zs, zt); loss = lc + LAM*lm
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(student.parameters(), 5.)
            scaler.step(opt); scaler.update(); sch.step()
            agg += np.array([loss.item(), lc.item(), lm.item()])*y.size(0); n += y.size(0)
        teacher.update(student)
        print(f"  c{cyc+1} {s['flag']:<18} loss {agg[0]/n:.4f} cls {agg[1]/n:.4f} "
              f"cons {agg[2]/n:.4f}", flush=True)

    r = {s["flag"]: score(teacher.t, VA[s["id"]], s) for s in SPECS}
    improved = []
    for s in SPECS:
        f_ = s["flag"]
        if r[f_]["auc"] > BEST_AUC[f_]:
            BEST_AUC[f_] = r[f_]["auc"]
            torch.save(snap(teacher.t), f"{OUT}/best_{f_}.pt")
            improved.append(f_.replace("mnist3d",""))
    mean = float(np.nanmean([v["auc"] for v in r.values()]))
    zmin = min(v["zstd"] for v in r.values()); dt = time.time()-t0
    hist.append(dict(cycle=cyc+1, mean=mean, **{k: v["auc"] for k,v in r.items()}))
    print(f"= cycle {cyc+1}  mean val AUC {mean:.4f}  zstd {zmin:.4f}"
          f"{'  <-- COLLAPSED' if zmin < ZSTD_FLOOR else ''}"
          f"  [{dt:.0f}s, ETA {(CYCLES-cyc-1)*dt/60:.0f} min]")
    print("  " + "  ".join(f"{k.replace('mnist3d',''):<9}{v['auc']:.3f}" for k,v in r.items()))
    print(f"  best-so-far mean {np.mean(list(BEST_AUC.values())):.4f}"
          f"   improved: {', '.join(improved) or 'none'}", flush=True)
    torch.save(dict(s=student.state_dict(), t=teacher.t.state_dict(), o=opt.state_dict(),
                    sch=sch.state_dict(), sc=scaler.state_dict(),
                    cycle=cyc+1, hist=hist, best_auc=BEST_AUC), ck)

  c1 organmnist3d       loss 2.5063 cls 2.3432 cons 0.1631
  c1 nodulemnist3d      loss 1.0810 cls 0.9847 cons 0.0963
  c1 fracturemnist3d    loss 1.2166 cls 1.1910 cons 0.0257
  c1 adrenalmnist3d     loss 1.1324 cls 1.1254 cons 0.0070
  c1 vesselmnist3d      loss 1.3168 cls 1.3008 cons 0.0159
  c1 synapsemnist3d     loss 0.3873 cls 0.3771 cons 0.0102
= cycle 1  mean val AUC 0.6967  zstd 0.0005  <-- COLLAPSED  [180s, ETA 117 min]
  organ    0.819  nodule   0.724  fracture 0.601  adrenal  0.703  vessel   0.773  synapse  0.560
  best-so-far mean 0.6967   improved: organ, nodule, fracture, adrenal, vessel, synapse
  c2 organmnist3d       loss 2.2955 cls 2.1959 cons 0.0996
  c2 nodulemnist3d      loss 1.0336 cls 0.9956 cons 0.0381
  c2 fracturemnist3d    loss 1.1525 cls 1.1383 cons 0.0142
  c2 adrenalmnist3d     loss 1.1139 cls 1.1027 cons 0.0111
  c2 vesselmnist3d      loss 1.2737 cls 1.2637 cons 0.0101
  c2 synapsemnist3d     loss 0.3771 cls 0.3706 cons 0.0065
= cycle 2  mean val AUC 0.7

## 10. Where each dataset peaked

The spread of peak cycles is itself a result: cyclic multi-task training does not
converge on all tasks at the same time, so a single stopping point is wrong for most of
them.

In [ ]:
print(f"{'dataset':<18}{'peak':>7}{'@cyc':>6}{'final':>8}{'lost':>8}{'base':>7}")
for s in SPECS:
    k = s["flag"]
    i = max(range(len(hist)), key=lambda j: hist[j][k])
    p, c, fin = hist[i][k], hist[i]["cycle"], hist[-1][k]
    print(f"{k:<18}{p:>7.3f}{c:>6}{fin:>8.3f}{fin-p:>+8.3f}{BASELINE[k]:>7.3f}")
mpk = np.mean([max(h[s['flag']] for h in hist) for s in SPECS])
mfin = np.mean([hist[-1][s['flag']] for s in SPECS])
print(f"\nbest-per-dataset mean {mpk:.4f} | final-cycle mean {mfin:.4f} "
      f"| recovered by per-task checkpointing {mpk-mfin:+.4f}")
json.dump(hist, open(f"{OUT}/history.json","w"), indent=2)

dataset              peak  @cyc   final    lost   base
organmnist3d        0.986    39   0.986  -0.000  0.996
nodulemnist3d       0.861    28   0.860  -0.001  0.863
fracturemnist3d     0.673    32   0.664  -0.008  0.712
adrenalmnist3d      0.827    40   0.827  +0.000  0.827
vesselmnist3d       0.835    32   0.834  -0.001  0.874
synapsemnist3d      0.622    23   0.596  -0.026  0.820

best-per-dataset mean 0.8006 | final-cycle mean 0.7946 | recovered by per-task checkpointing +0.0060


## 11. Linear probe

Each dataset loads **its own** best teacher. Flip-TTA embeddings, standardised features,
`C` selected on validation and reported on test. Both the encoder feature and the
projection are probed and the better is picked on validation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def load_best(flag):
    p = f"{OUT}/best_{flag}.pt"
    if not os.path.exists(p): return False
    teacher.t.load_state_dict(torch.load(p, map_location=dev)); return True

@torch.no_grad()
def embed(model, loader, sid, tta=True):
    model.eval(); Fs, Ps, Y = [], [], []
    for xb, y in loader:
        x = to_gpu(xb, sid)
        views = [x, x.flip(2), x.flip(3), x.flip(4)] if tta else [x]
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type=="cuda"):
            f = torch.stack([model.feat(v).float() for v in views]).mean(0)
            p = model.proj(f).float()
        Fs.append(f.cpu().numpy()); Ps.append(p.cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(Fs), np.concatenate(Ps), np.concatenate(Y)

def fit_probe(Xtr, ytr, Xva, yva, Xte, yte, spec):
    sc = StandardScaler().fit(Xtr)
    A, B, C_ = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
    def auc(clf, X, y):
        p = np.zeros((len(y), spec["n"])); p[:, clf.classes_] = clf.predict_proba(X)
        return roc_auc_score(y, p[:,1]) if spec["binary"] else ovr_auc(y, p, spec["n"])
    bv, bc = -1, None
    for C in (1e-3, 1e-2, 1e-1, 1.0):        # selected on VAL, never on test
        clf = LogisticRegression(max_iter=3000, C=C).fit(A, ytr)
        v = auc(clf, B, yva)
        if v > bv: bv, bc = v, clf
    return float(auc(bc, C_, yte)), bv

PROBE, JOINT = {}, {}
for s in SPECS:
    ok = load_best(s["flag"])
    JOINT[s["flag"]] = score(teacher.t, TE[s["id"]], s, tta=True)["auc"]
    E = {sp: embed(teacher.t, L[s["id"]], s["id"])
         for sp, L in (("tr",TR), ("va",VA), ("te",TE))}
    cand = {nm: fit_probe(E["tr"][j], E["tr"][2], E["va"][j], E["va"][2],
                          E["te"][j], E["te"][2], s) for j, nm in ((0,"feat"), (1,"proj"))}
    pick = max(cand, key=lambda k: cand[k][1])     # feat-vs-proj chosen on VAL
    PROBE[s["flag"]] = cand[pick][0]
    print(f"{s['flag']:<18} probe {cand[pick][0]:.4f} (via {pick}; "
          f"feat {cand['feat'][0]:.4f} / proj {cand['proj'][0]:.4f})"
          f"{'' if ok else '  [no best_*.pt - used last teacher]'}", flush=True)
print(f"\nprobe mean {np.nanmean(list(PROBE.values())):.4f}")

organmnist3d       probe 0.9857 (via feat; feat 0.9857 / proj 0.9750)
nodulemnist3d      probe 0.8852 (via proj; feat 0.8890 / proj 0.8852)
fracturemnist3d    probe 0.6724 (via feat; feat 0.6724 / proj 0.6699)
adrenalmnist3d     probe 0.8718 (via proj; feat 0.8672 / proj 0.8718)
vesselmnist3d      probe 0.9013 (via feat; feat 0.9013 / proj 0.8350)
synapsemnist3d     probe 0.7216 (via proj; feat 0.7303 / proj 0.7216)

probe mean 0.8396


## 12. Fine-tuning — the fair comparison

Methodology §5.1's second configuration. Identical to cell 8 except the encoder and
projector start from that dataset's best cyclically-pretrained **teacher** rather than
the raw SSL checkpoint.

`ft − solo` is the clean Ark+ transfer measurement: same architecture, same budget, same
data, only the initialisation differs. `ft − base` is the honest comparison against the
official numbers, which are fully-trained supervised models.

In [ ]:
FT = {}
for s in SPECS:
    t0 = time.time(); load_best(s["flag"])
    FT[s["flag"]] = train_one(s, SOLO_EPOCHS, init=teacher.t.state_dict(),
                              lr=1e-3, enc_mult=0.1)
    print(f"{s['flag']:<18} finetune {FT[s['flag']]['auc']:.4f}  "
          f"ACC {FT[s['flag']]['acc']:.4f}  [{time.time()-t0:.0f}s]", flush=True)
json.dump(FT, open(f"{OUT}/finetune.json","w"), indent=2)

/tmp/ipykernel_918/780145861.py:23: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


organmnist3d       finetune 0.9734  ACC 0.7295  [709s]


/tmp/ipykernel_918/780145861.py:23: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


nodulemnist3d      finetune 0.9028  ACC 0.8290  [828s]


/tmp/ipykernel_918/780145861.py:23: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


...

adrenalmnist3d     finetune 0.8826  ACC 0.7886  [827s]


/tmp/ipykernel_918/780145861.py:23: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


vesselmnist3d      finetune 0.8858  ACC 0.7880  [953s]


/tmp/ipykernel_918/780145861.py:23: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


synapsemnist3d     finetune 0.7693  ACC 0.7528  [880s]


## 13. Results

In [ ]:
hdr = (f"{'dataset':<18}{'solo':>8}{'joint':>8}{'probe':>8}{'finetune':>10}"
       f"{'base':>8}{'ft-solo':>9}{'ft-base':>9}")
print(hdr); print("-"*len(hdr))
rows = []
for s in SPECS:
    f_ = s["flag"]
    so, jo, pr = SOLO[f_]["auc"], JOINT[f_], PROBE[f_]
    ft, bl = FT[f_]["auc"], BASELINE[f_]
    rows.append((f_, so, jo, pr, ft, bl))
    print(f"{f_:<18}{so:>8.4f}{jo:>8.4f}{pr:>8.4f}{ft:>10.4f}{bl:>8.3f}"
          f"{ft-so:>+9.4f}{ft-bl:>+9.4f}")
M = lambda i: float(np.nanmean([r[i] for r in rows]))
print("-"*len(hdr))
print(f"{'MEAN':<18}{M(1):>8.4f}{M(2):>8.4f}{M(3):>8.4f}{M(4):>10.4f}{M(5):>8.3f}"
      f"{M(4)-M(1):>+9.4f}{M(4)-M(5):>+9.4f}")

print(f"\nArk+ transfer, ft > solo on every dataset: {all(r[4] > r[1] for r in rows)}")
print(f"beats official baseline: {[r[0] for r in rows if r[4] >= r[5]] or 'none'}")
json.dump([dict(zip(("dataset","solo","joint","probe","finetune","baseline"), r))
           for r in rows], open(f"{OUT}/results.json","w"), indent=2)

dataset               solo   joint   probe  finetune    base  ft-solo  ft-base
------------------------------------------------------------------------------
organmnist3d        0.9713  0.9645  0.9857    0.9734   0.996  +0.0021  -0.0226
nodulemnist3d       0.8662  0.8738  0.8852    0.9028   0.863  +0.0365  +0.0398
fracturemnist3d     0.6149  0.6755  0.6724    0.7005   0.712  +0.0856  -0.0115
adrenalmnist3d      0.8008  0.8684  0.8718    0.8826   0.827  +0.0818  +0.0556
vesselmnist3d       0.7826  0.7972  0.9013    0.8858   0.874  +0.1032  +0.0118
synapsemnist3d      0.7066  0.6870  0.7216    0.7693   0.820  +0.0627  -0.0507
------------------------------------------------------------------------------
MEAN                0.7904  0.8111  0.8396    0.8524   0.849  +0.0620  +0.0037

Ark+ transfer, ft > solo on every dataset: True
beats official baseline: ['nodulemnist3d', 'adrenalmnist3d', 'vesselmnist3d']
